In [67]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# 1. Load the Worcester datasets
# Ensure these files are in your current working directory
pois = pd.read_csv('worcester_pois.csv')
visits = pd.read_csv('worcester_cbg_poi_visits.csv')
distances = pd.read_csv('worcester_cbg_poi_distance.csv')

# 2. Global Pre-cleaning
# Remove zeros to prevent errors in log/power operations
pois = pois[pois['wkt_area_sq_meters'] > 0]
distances = distances[distances['distance_m'] > 0]
visits = visits[visits['visit_count'] > 0]

def calibrate_huff_params_final(category_name):
    # Filter POIs by specific category
    cat_poi_data = pois[pois['top_category'] == category_name]
    cat_pois = cat_poi_data[['placekey', 'wkt_area_sq_meters']]
    cat_pks = cat_pois['placekey'].unique()
    
    # Retrieve the primary NAICS code for this category
    naics = cat_poi_data['naics_code'].mode()[0] if not cat_poi_data['naics_code'].empty else "Unknown"
    
    # Get actual visits to these locations
    cat_visits = visits[visits['placekey'].isin(cat_pks)]
    
    # FILTER: Categories must have more than 50 historical records
    if len(cat_visits) <= 50:
        return None
    
    # Calculate Observed Market Share (Pij_actual)
    cbg_totals = cat_visits.groupby('visitor_home_cbg')['visit_count'].sum().reset_index()
    cbg_totals.columns = ['visitor_home_cbg', 'total_cat_visits']
    
    actual_shares = cat_visits.merge(cbg_totals, on='visitor_home_cbg')
    actual_shares['pij_actual'] = actual_shares['visit_count'] / actual_shares['total_cat_visits']
    
    # Identify the Global Competitive Set (The Denominator)
    # We look at all potential POIs in the category for the neighborhoods that visited
    active_cbgs = cat_visits['visitor_home_cbg'].unique()
    competitive_set = distances[
        (distances['GEOID10'].isin(active_cbgs)) & 
        (distances['placekey'].isin(cat_pks))
    ].merge(cat_pois, on='placekey')
    
    # Pre-calculate logs for computational speed
    competitive_set['ln_area'] = np.log(competitive_set['wkt_area_sq_meters'])
    competitive_set['ln_dist'] = np.log(competitive_set['distance_m'])
    
    # Grid Search Parameters
    best_a, best_b, max_corr = 1.0, 1.0, -1
    search_range = np.round(np.arange(1.0, 3.1, 0.1), 1)
    
    # Iterate through Alpha and Beta combinations
    for a in search_range:
        # Pre-compute the attractiveness component
        competitive_set['attr'] = np.exp(a * competitive_set['ln_area'])
        
        for b in search_range:
            # Calculate Utility: U = (Area^alpha) / (Dist^beta)
            competitive_set['u'] = competitive_set['attr'] / np.exp(b * competitive_set['ln_dist'])
            
            # Sum utilities per neighborhood (The Denominator)
            denom = competitive_set.groupby('GEOID10')['u'].sum().reset_index()
            denom.columns = ['GEOID10', 'sum_u']
            
            # Calculate Predicted Probability
            pred = competitive_set.merge(denom, on='GEOID10')
            pred['pij_pred'] = pred['u'] / pred['sum_u']
            
            # Join with actual shares to evaluate performance
            comparison = actual_shares.merge(
                pred[['GEOID10', 'placekey', 'pij_pred']], 
                left_on=['visitor_home_cbg', 'placekey'], 
                right_on=['GEOID10', 'placekey']
            )
            
            if len(comparison) < 2: continue
            
            # Evaluate using Pearson Correlation
            corr, _ = pearsonr(comparison['pij_actual'], comparison['pij_pred'])
            
            if corr > max_corr:
                max_corr = corr
                best_a, best_b = a, b
    
    # FILTER: Only keep results with Correlation > 0.25
    if max_corr <= 0.25:
        return None
                
    return {
        'top_category': category_name, 
        'NAICS code': naics,
        'alpha': best_a, 
        'beta': best_b, 
        'correlation': round(max_corr, 4)
    }

# 3. Main Execution Loop
all_categories = pois['top_category'].unique()
final_results = []

print(f"Calibrating {len(all_categories)} business categories...")

for cat in all_categories:
    result = calibrate_huff_params_final(cat)
    if result:
        final_results.append(result)

# 4. Generate Final DataFrame and CSV
final_df = pd.DataFrame(final_results)
#final_df.to_csv('calibrated_parameters_filtered.csv', index=False)

print("\n--- Calibration Successful ---")
print(f"Total High-Confidence Categories Found: {len(final_df)}")
print(final_df.head(10))

Calibrating 130 business categories...

--- Calibration Successful ---
Total High-Confidence Categories Found: 23
                                     top_category  NAICS code  alpha  beta  \
0          Building Material and Supplies Dealers        4441    1.9   1.0   
1             Bakeries and Tortilla Manufacturing      311811    3.0   1.0   
2               Other Miscellaneous Manufacturing        3399    3.0   1.0   
3                               Gasoline Stations      447110    1.9   1.0   
4                             Offices of Dentists      621210    1.6   1.0   
5     Activities Related to Credit Intermediation      522310    2.1   2.3   
6    Justice, Public Order, and Safety Activities      922110    3.0   1.6   
7             Other Miscellaneous Store Retailers      453991    3.0   1.0   
8  Automotive Parts, Accessories, and Tire Stores      441310    1.7   1.0   
9                   Beer, Wine, and Liquor Stores      445310    1.5   1.6   

   correlation  
0       0.